# Model tournament on the DGX A100 (80 GB)
Run on **JupyterHub**. Follow `docs/DGX_SETUP.md` first (one-time setup: environment, token, data, tokenizer).
Rules baked in: **never GPU 7**; long jobs run in the background with `nohup` (a closed tab must not kill them); results are copied off the machine;
at the end clear the kernel memory and **Stop My Server** (File > Hub Control Panel).

Run the cells in order. Each cell says what to look for.

In [ ]:
# ---------- 1. SETTINGS (edit) ----------
REPO_DIR     = "~/qwenlab"                       # where you unpacked qwenlab v0.4
TOKENIZER    = "~/qwenlab/tokenizers/malayalam_unigram32k.model"
HF_REPO      = "Akzhai/malayalam-lm-tournament"
SUBDIR       = "dgx"                              # folder inside the HF repo for this machine
PLAN_NAME    = "race1_screen"                     # race0_lr | race1_screen | race2_finalists | race3_batch | race3_context
N_GPUS       = 6                                  # how many GPUs you may use right now (free ones among 0-6)
BACKUP_CHECKPOINTS = False                        # True for the long finals (race2_finalists)

In [ ]:
# ---------- 2. Environment ----------
import os, subprocess, json, getpass, shutil
REPO_DIR = os.path.expanduser(REPO_DIR); TOKENIZER = os.path.expanduser(TOKENIZER)
os.chdir(REPO_DIR)
os.environ["HF_HOME"] = os.path.expanduser("~/hf_cache")           # keep caches inside your own home directory
# Hugging Face token: typed once, kept only in this kernel's memory (never saved in the notebook). If you did `export HF_TOKEN=...`
# in the terminal before starting JupyterHub it is picked up instead.
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF token (fine-grained, write access to the tournament repo only): ")
from huggingface_hub import HfApi
print("HF user:", HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"])
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
# ---------- 3. Pick free GPUs among 0-6 (GPU 7 is reserved) and build the run plan ----------
out = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
free = sorted([(int(m), int(i)) for i, m in (l.split(",") for l in out.strip().splitlines()) if int(i) != 7])
GPUS = [i for m, i in free if m < 2000][:N_GPUS]          # GPUs using less than 2 GB are considered free
print("GPUs I will use:", GPUS, "| busy:", [(i, m) for m, i in free if m >= 2000])
assert GPUS and 7 not in GPUS, "no free GPU among 0-6: ask the other users / wait"
GPU_ARG = ",".join(map(str, GPUS))

plan = json.load(open(f"configs/tournament/{PLAN_NAME}.json", encoding="utf-8"))
plan["overrides"] += [f'tokenizer.path="{TOKENIZER}"']
plan["gpus"] = GPUS
if BACKUP_CHECKPOINTS:
    plan["hub_repo"] = HF_REPO
PLAN_RUN = f"{REPO_DIR}/plan_{PLAN_NAME}_run.json"
json.dump(plan, open(PLAN_RUN, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print("plan:", PLAN_NAME, "| candidates:", [c["id"] for c in plan["candidates"]], "| tokens/run:", f"{plan['tokens']:,}")
def tour(phase, extra=""):
    return f'cd {REPO_DIR} && python scripts/tournament.py --plan {PLAN_RUN} --phase {phase} {extra}'

## 4. Tests on the DGX (bf16 + real CUDA)  -  *Pass when:* all pass

In [ ]:
!cd {REPO_DIR} && python -m pytest tests -q -x

## 5. Check every candidate at full size  -  *Pass when:* all `status: ok`, parity MATCH

In [ ]:
!{tour("check")}

## 6. Build the shared data cache ONCE (can take 1-2 h: runs in the background; the log shows progress)

In [ ]:
p = subprocess.Popen(tour("prepare"), shell=True, stdout=open("prepare.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
print("started, pid", p.pid, "-> follow with: tail -f prepare.log")

In [ ]:
!tail -n 15 prepare.log

## 7. Probe (about 1 h on ONE free GPU)
tokens/s, peak memory, best micro-batch and longest context for every candidate. Wait until `prepare.log` has finished (`train tokens available: ...`).

In [ ]:
!{tour("probe", "--gpus " + str(GPUS[0]))}

**Budget:** how many tokens per run fit in 5 hours with these GPUs. If it is far from the `tokens` value in the plan, edit `tokens` in `configs/tournament/<plan>.json` (a multiple of `batch_tokens`) and re-run cell 3.

In [ ]:
!{tour("budget", "--hours 5 --gpus " + GPU_ARG)}

## 8. RACE in the background
The race keeps running if you close the browser tab. Re-running this cell is safe: finished runs are skipped, unfinished runs resume from their last checkpoint (every 500 steps).

In [ ]:
race = subprocess.Popen(tour("race", "--gpus " + GPU_ARG), shell=True, stdout=open("race.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
print("race started, pid", race.pid, "-> follow with: tail -f race.log")

In [ ]:
# Progress at any time (re-run this cell): last log lines per run + GPU load
!tail -n 6 race.log
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv
!for f in runs/tournament/{PLAN_NAME}/*/metrics.jsonl; do echo "$$f: $$(tail -n 1 $$f | cut -c1-160)"; done

## 9. Report (works while the race runs: it ranks what has finished so far)

In [ ]:
!{tour("report")}

In [ ]:
# Upload the findings (report, probe, run metrics/configs/logs; no checkpoints). Run after every race.
!cd {REPO_DIR} && python scripts/upload_results.py --plan {PLAN_RUN} --repo {HF_REPO} --subdir {SUBDIR}

## 10. When the race is finished
1. Findings uploaded (previous cell) and `runs/tournament/<plan>/report.md` read.
2. Free the disk: delete the checkpoints (about 6 GB per run) once `results.json` exists:
   `find runs/tournament -name checkpoints -type d -prune -exec rm -rf {} +`
3. Next plan (see `docs/TOURNAMENT_RUNBOOK.md`, section C): edit `PLAN_NAME` in cell 1 and re-run from cell 3.
4. **Last cell below** clears the kernel memory (DGX rule). Then File > Hub Control Panel > **Stop My Server**.

In [ ]:
# DGX guideline: clear memory at the end of your program (this shuts this notebook's kernel down).
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)